# 06 — AOI inference: super-resolved imagery for a chosen region and month

General version of notebook 5: instead of one fixed pre-selected scene, this picks **any named sub-national region + any month**, finds the best available Sentinel-2 scene(s) covering it, runs the trained model, and mosaics the result into one output GeoTIFF (a region typically spans more than one Sentinel-2 tile, unlike notebook 5's single-scene case).

**First real test case: Ghana's Western Region, January 2026.** Chosen deliberately as a hard case — it's one of Ghana's most persistently cloud-affected regions (humid forest zone), so this also stress-tests whether a genuinely usable scene exists at all for a given month, not just whether the pipeline runs.

**Scope caveat**: the model was trained only on Ghana dry-season (Jan–May 2025) imagery — see `agents.md` "Scope decision"/"Study area". This notebook stays within Ghana; applying it outside Ghana, or to a very different season, is untested.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import rasterio
import torch
from tqdm.auto import tqdm

sys.path.insert(0, str(Path.cwd().parent / "src"))

from s2sr import boundaries, config, quality, raster, stac
from s2sr.inference import mosaic_geotiffs, superresolve_scene
from s2sr.model import DSen2Net20m

In [ ]:
# Change these for a different region/month.
REGION_NAME = "Western Region, Ghana"
REGION_SLUG = "western_region"  # used in cached filenames below
DATE_START = "2026-01-01"
DATE_END = "2026-01-31"

region_boundary_path = config.REPO_ROOT / "data" / "boundaries" / f"ghana_{REGION_SLUG}.geojson"
region_boundary = boundaries.fetch_boundary(REGION_NAME, region_boundary_path, feature_type="state")
region_bbox = tuple(region_boundary.total_bounds)

print(f"{REGION_NAME}: bbox={region_bbox}")
region_boundary.plot(edgecolor="black", facecolor="none")

## Find candidate scenes

Same two-stage AOI approach as notebook 1: a generous bbox search, then tightened against the region's real boundary — see `agents.md` "Study area" for why this is two separate steps rather than an `intersects=<polygon>` search.

In [ ]:
items = stac.search_scenes(bbox=region_bbox, datetime_range=f"{DATE_START}/{DATE_END}")
items_by_id = {item.id: item for item in items}
scenes_all = stac.items_to_dataframe(items)

scenes = stac.filter_by_aoi(scenes_all, aoi_path=region_boundary_path)
print(f"{len(scenes_all)} scenes found, {len(scenes)} after tightening to the real region boundary")
scenes.groupby("mgrs_tile").size()

## Per-scene quality (no filtering yet — just reporting)

This region is a genuinely hard case for imagery clarity, so rather than silently dropping anything that fails the usual Ghana-wide thresholds (`config.MAX_SCL_CLOUD_SHADOW_FRACTION` etc.), this computes and shows the real numbers for every candidate first — the best *available* scene per tile gets used regardless of whether it clears those thresholds, with a warning if it doesn't, so the output's real quality is visible rather than assumed.

In [ ]:
records = []
for _, row in tqdm(scenes.iterrows(), total=len(scenes)):
    item = items_by_id[row["id"]]
    scl = quality.scl_stats(item, region_boundary)
    records.append(
        {
            "id": row["id"],
            "mgrs_tile": row["mgrs_tile"],
            "datetime": row["datetime"],
            "cloud_shadow_fraction": scl["cloud_shadow_fraction"],
            "nodata_fraction": scl["nodata_fraction"],
        }
    )

quality_df = pd.DataFrame(records)
quality_df.sort_values(["mgrs_tile", "cloud_shadow_fraction"])

In [ ]:
best_per_tile = (
    quality_df.sort_values("cloud_shadow_fraction")
    .groupby("mgrs_tile", as_index=False)
    .first()
)

for _, row in best_per_tile.iterrows():
    flag = " ** ABOVE NORMAL THRESHOLD **" if row["cloud_shadow_fraction"] > config.MAX_SCL_CLOUD_SHADOW_FRACTION else ""
    print(
        f"{row['mgrs_tile']}: {row['id']}  "
        f"cloud/shadow={row['cloud_shadow_fraction']:.3f}  nodata={row['nodata_fraction']:.3f}{flag}"
    )

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DSen2Net20m(guide_channels=len(config.BANDS_10M), target_channels=len(config.BANDS_20M)).to(device)
checkpoint = torch.load(config.REPO_ROOT / "models" / "dsen2_20m.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Device: {device}, checkpoint epoch {checkpoint['epoch'] + 1}, val_loss={checkpoint['val_loss']:.4f}")

In [ ]:
out_dir = config.REPO_ROOT / "data" / "processed" / "superresolved" / f"{REGION_SLUG}_{DATE_START[:7]}"
tile_paths = []
for _, row in tqdm(best_per_tile.iterrows(), total=len(best_per_tile)):
    item = items_by_id[row["id"]]
    tile_out_path = out_dir / f"{row['mgrs_tile']}.tif"
    superresolve_scene(item, region_boundary, model, device, tile_out_path)
    tile_paths.append(tile_out_path)

print(f"{len(tile_paths)} tile(s) processed")

In [ ]:
mosaic_path = config.REPO_ROOT / "data" / "processed" / "superresolved" / f"{REGION_SLUG}_{DATE_START[:7]}.tif"
mosaic_geotiffs(tile_paths, mosaic_path)
print(f"Saved: {mosaic_path}")

In [ ]:
with rasterio.open(mosaic_path) as src:
    band_idx = config.BANDS_20M.index("B11") + 1  # rasterio band indices are 1-based
    b11 = src.read(band_idx)

plt.figure(figsize=(8, 8))
plt.imshow(raster.percentile_stretch(b11), cmap="gray")
plt.title(f"{REGION_NAME} — super-resolved B11, {DATE_START[:7]}")
plt.axis("off")

## Next steps

- Check the quality report above — if every tile's best available scene is well above `MAX_SCL_CLOUD_SHADOW_FRACTION`, that's a real finding about this region/month's imagery availability, not a pipeline problem. Widening `DATE_START`/`DATE_END` (e.g. the whole dry season instead of one month) is the standard fix.
- The mosaic only covers `patch_size`-aligned tiles fully inside each scene's extent — same edge limitation as notebook 3/5. Small gaps at scene/tile boundaries are expected, not a bug.
- To run this for a different region, change `REGION_NAME`/`REGION_SLUG` at the top — any Nominatim-resolvable Ghana region name should work, since the whole pipeline below is AOI-agnostic.